# 🧬 D-MorphNet — خط الأنابيب الكامل في ملف واحد
### كشف دمج الوجوه: EfficientNet-B6 + SVM — من بناء البيانات حتى التنبؤ الفوري

هذا الدفتر **يدمج الدفاتر العشرة كلها** في ملف واحد مرتب بالأجزاء، بعد مراجعة الكود وإصلاح المشاكل، وأهم تحسين: **ملف الهويات `identity_CelebA.txt` يُجلَب تلقائياً** — لا حاجة لأي رفع يدوي.

| الجزء | المحتوى |
|---|---|
| ١ | الإعدادات العامة (وضع تجريبي سريع / كامل) |
| ٢ | تحميل البيانات + الهويات **تلقائياً** |
| ٣ | توليد صور المورف + المراجعة + التقسيم العادل |
| ٤ | المعالجة المسبقة (528×528 + CLAHE) وزيادة البيانات |
| ٥ | استخراج السمات بـ EfficientNet-B6 (+ ضبط دقيق اختياري) |
| ٦ | تدريب SVM والتحقق من دالة القرار |
| ٧ | النتائج: مصفوفة الالتباس + المقاييس + ROC/AUC |
| ٨ | تحسين العتبة |
| ٩ | مقارنة المعماريات (اختياري) |
| ١٠ | التنبؤ الفوري: رفع صور + تعدد وجوه + قياس السرعة |
| ١١ | الحفظ في Drive (اختياري) والخلاصة |

> ✅ **شغّل الخلايا بالترتيب من الأعلى للأسفل** — في الوضع التجريبي `DEMO = True` يكتمل كل شيء خلال ~30–45 دقيقة وترى **نتيجة في كل خلية**. بعد نجاح التجربة غيّر `DEMO = False` لأرقام الورقة الكاملة (45,000 صورة — عدة ساعات).
>
> ⚙️ فعّل GPU: ‏Runtime ← Change runtime type ← **GPU (T4)**.


## الجزء ١ — الإعدادات العامة

كل مفاتيح التحكم في مكان واحد:
- `DEMO`: تشغيل تجريبي سريع بأعداد مصغّرة (نفس نِسَب الورقة) لرؤية النتائج في كل خلية.
- `DO_FINETUNE`: الضبط الدقيق لطبقات B6 العليا (اختياري — يضيف وقتاً).
- `RUN_COMPARISONS`: مقارنة B0/B5/B6 (الجدول ٢) — اختياري.
- `SAVE_TO_DRIVE`: حفظ النماذج والنتائج في Google Drive في النهاية.


In [ ]:
# ================== مفاتيح التحكم ==================
DEMO = True                # True = تجربة سريعة | False = أرقام الورقة كاملة
DO_FINETUNE = False        # ضبط دقيق لطبقات B6 العليا (الجزء 5)
RUN_COMPARISONS = False    # مقارنة B0/B5/B6 — الجدول 2 (الجزء 9)
SAVE_TO_DRIVE = False      # حفظ كل شيء في Drive في النهاية (الجزء 11)

# ================== أعداد الصور ==================
if DEMO:
    SPLITS = {                                   # نفس نسب الورقة مصغّرة 1/40
        "train": {"real": 400, "morph": 350},
        "val":   {"real": 175, "morph": 150},
        "test":  {"real": 50,  "morph": 50},
    }
else:
    SPLITS = {                                   # أرقام الورقة (45,000 صورة)
        "train": {"real": 16000, "morph": 14000},
        "val":   {"real": 7000,  "morph": 6000},
        "test":  {"real": 1000,  "morph": 1000},
    }

# ================== الثوابت ==================
SEED = 42
MORPH_SIZE = (256, 256)      # حجم توليد المورف (الجزء 3)
INPUT_SIZE = 528             # حجم إدخال EfficientNet-B6 (الأجزاء 4+)
MORPH_ALPHA = 0.5            # نسبة مزج الوجهين
MIN_SHARPNESS = 20.0         # حد المراجعة التلقائية (حدة لابلاس)
JPEG_QUALITY = 90            # جودة الضغط الموحدة

BASE = "/content/DMorphNet"
RAW_DIR  = f"{BASE}/dataset"      # المجموعة المولدة (الجزء 3)
PROC_DIR = f"{BASE}/processed"    # بعد المعالجة المسبقة (الجزء 4)
FEAT_DIR = f"{BASE}/features"     # السمات المستخرجة (الجزء 5)

import os, glob, random, shutil, time
import numpy as np

os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"   # يقلل مشاكل ذاكرة GPU
random.seed(SEED)
np.random.seed(SEED)

for d in [RAW_DIR, PROC_DIR, FEAT_DIR]:
    os.makedirs(d, exist_ok=True)
CLASSES = {"real": 0, "morph": 1}
SPLIT_NAMES = list(SPLITS)

t_real  = sum(v["real"]  for v in SPLITS.values())
t_morph = sum(v["morph"] for v in SPLITS.values())
print(f"الوضع: {'تجريبي سريع 🚀' if DEMO else 'كامل (أرقام الورقة) 🐢'}")
print(f"الهدف: {t_real} حقيقية + {t_morph} مدموجة = {t_real + t_morph} صورة")
for s, v in SPLITS.items():
    print(f"  {s}: حقيقية={v['real']}  مدموجة={v['morph']}")


### تثبيت المكتبات


In [ ]:
!pip -q install kagglehub mediapipe opencv-python-headless scipy tqdm pandas scikit-learn joblib gdown

import cv2, tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

tf.random.set_seed(SEED)
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass
print("TensorFlow:", tf.__version__, "| OpenCV:", cv2.__version__)
print("GPU:", gpus if gpus else "⚠️ لا يوجد GPU — فعّله: Runtime ← Change runtime type ← GPU")


## الجزء ٢ — تحميل الصور الحقيقية وملف الهويات (تلقائياً بالكامل)

### ٢.١ تحميل صور CelebA (نفس صور FaceMorph_EClub_Task — 202,599 صورة)


In [ ]:
import kagglehub

DATA_ROOT = kagglehub.dataset_download("jessicali9530/celeba-dataset")
all_jpgs = glob.glob(os.path.join(DATA_ROOT, "**", "*.jpg"), recursive=True)
IMG_DIR = os.path.dirname(all_jpgs[0])

print("مسار التحميل:", DATA_ROOT)
print("عدد الصور الحقيقية المتاحة:", len(all_jpgs))
print("مجلد الصور:", IMG_DIR)


### ٢.٢ ملف الهويات — بلا أي خطوة يدوية ✋🚫

نجرّب ثلاث طرق **بالترتيب تلقائياً**:
1. البحث عن أي ملف `identity` داخل ما حُمّل من Kaggle.
2. تنزيل `identity_CelebA.txt` الرسمي من رابط Google Drive العام لمجموعة CelebA عبر `gdown`.
3. **الخطة الاحتياطية المضمونة**: اعتبار كل صورة هوية مستقلة (Pseudo-Identity) — كل شيء يعمل، مع تنبيه أن ضمان فصل الهويات يصبح على مستوى الصورة لا الشخص.


In [ ]:
identity_path = None

# الطريقة 1: البحث داخل الملفات المحملة
for cand in glob.glob(os.path.join(DATA_ROOT, "**", "*identity*"), recursive=True):
    if os.path.isfile(cand):
        identity_path = cand
        print("✅ وُجد ملف الهويات داخل بيانات Kaggle:", cand)
        break

# الطريقة 2: التنزيل التلقائي من الرابط الرسمي العام لـ CelebA
if identity_path is None:
    try:
        import gdown
        out = "/content/identity_CelebA.txt"
        gdown.download(id="1_ee_0u7vcNLOfNLegJRHmolfH5ICW-XS",
                       output=out, quiet=True)
        if os.path.exists(out) and os.path.getsize(out) > 1_000_000:
            identity_path = out
            print("✅ تم تنزيل identity_CelebA.txt تلقائياً من المصدر الرسمي")
    except Exception as e:
        print("تعذّر التنزيل التلقائي:", e)

# قراءة الملف أو تفعيل الخطة الاحتياطية
if identity_path:
    if identity_path.endswith(".txt"):
        ident = pd.read_csv(identity_path, sep=r"\s+", header=None,
                            names=["image_id", "identity"])
    else:
        ident = pd.read_csv(identity_path)
        ident.columns = ["image_id", "identity"]
    print(f"عدد الصور: {len(ident)} — عدد الأشخاص: {ident['identity'].nunique()}")
else:
    # الطريقة 3: هويات افتراضية (كل صورة = شخص مستقل) — يعمل دائماً
    names = sorted(os.path.basename(p) for p in all_jpgs)
    ident = pd.DataFrame({"image_id": names,
                          "identity": np.arange(len(names))})
    print("⚠️ لم يتوفر ملف الهويات — تم تفعيل الهويات الافتراضية تلقائياً")
    print("   (الفصل بين الأقسام سيكون على مستوى الصورة، وكل الخطوات ستعمل)")

ident.head()


## الجزء ٣ — بناء المجموعة وتوليد صور المورف

### ٣.١ تقسيم الهويات واختيار الصور الحقيقية

كل شخص يذهب بكامل صوره إلى قسم واحد فقط (تدريب أو تحقق أو اختبار) — ثم تُولَّد المورفات من أشخاص داخل نفس القسم فقط، فلا تتسرب أي هوية بين الأقسام.


In [ ]:
by_id = ident.groupby("identity")["image_id"].apply(list).to_dict()
img2id = dict(zip(ident["image_id"], ident["identity"]))

available = {os.path.basename(p) for p in all_jpgs}
by_id = {k: [f for f in v if f in available] for k, v in by_id.items()}
by_id = {k: v for k, v in by_id.items() if v}

identities = list(by_id)
random.shuffle(identities)

selected, id_iter = {}, iter(identities)
for split, tgt in SPLITS.items():
    images, sids = [], set()
    while len(images) < tgt["real"]:
        pid = next(id_iter, None)
        if pid is None:                     # حارس: نفدت الهويات
            raise RuntimeError("عدد الهويات لا يكفي الأهداف المطلوبة")
        sids.add(pid)
        images.extend(by_id[pid])
    selected[split] = {"identities": sids, "images": images[: tgt["real"]]}
    print(f"{split}: {len(selected[split]['images'])} صورة حقيقية "
          f"من {len(sids)} هوية")

for a in SPLIT_NAMES:
    for b in SPLIT_NAMES:
        if a < b:
            assert not (selected[a]["identities"] & selected[b]["identities"])
print("✅ الهويات منفصلة تماماً بين الأقسام الثلاثة")


### ٣.٢ دوال توليد المورف (بأسلوب AI FaceSwap)

استخراج 468 معلماً بـ MediaPipe ← معالم وسطية ← تقسيم Delaunay مثلثي ← تشويه Affine لكل مثلث ← مزج الوجهين 50/50.


In [ ]:
import mediapipe as mp
from scipy.spatial import Delaunay

# واجهة موحدة تعمل مع نسختي MediaPipe:
# القديمة (mp.solutions) والجديدة (Tasks API) التي أُزيلت منها solutions
if hasattr(mp, "solutions"):
    _face_mesh = mp.solutions.face_mesh.FaceMesh(
        static_image_mode=True, max_num_faces=1, min_detection_confidence=0.5)

    def _mesh_points(img_rgb):
        res = _face_mesh.process(img_rgb)
        if not res.multi_face_landmarks:
            return None
        return res.multi_face_landmarks[0].landmark
else:
    import urllib.request
    from mediapipe.tasks.python import BaseOptions
    from mediapipe.tasks.python import vision as mp_vision
    _LM_MODEL = "/content/face_landmarker.task"
    if not os.path.exists(_LM_MODEL):
        urllib.request.urlretrieve(
            "https://storage.googleapis.com/mediapipe-models/face_landmarker/"
            "face_landmarker/float16/1/face_landmarker.task", _LM_MODEL)
    _landmarker = mp_vision.FaceLandmarker.create_from_options(
        mp_vision.FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=_LM_MODEL), num_faces=1))

    def _mesh_points(img_rgb):
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB,
                          data=np.ascontiguousarray(img_rgb))
        res = _landmarker.detect(mp_img)
        return res.face_landmarks[0] if res.face_landmarks else None

print("واجهة MediaPipe المستخدمة:",
      "القديمة (solutions)" if hasattr(mp, "solutions") else "الجديدة (Tasks)")


def get_landmarks(img_bgr):
    # معالم الوجه + 8 نقاط حواف؛ None إذا لم يوجد وجه
    lms = _mesh_points(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    if lms is None:
        return None
    h, w = img_bgr.shape[:2]
    pts = np.array([[lm.x * w, lm.y * h] for lm in lms], np.float32)
    h1, w1 = h - 1, w - 1
    border = np.array([[0, 0], [w1 // 2, 0], [w1, 0], [w1, h1 // 2], [w1, h1],
                       [w1 // 2, h1], [0, h1], [0, h1 // 2]], np.float32)
    pts = np.vstack([pts, border])
    pts[:, 0] = np.clip(pts[:, 0], 0, w1)
    pts[:, 1] = np.clip(pts[:, 1], 0, h1)
    return pts


def _morph_triangle(img1, img2, out, t1, t2, t, alpha):
    r1, r2, r = (cv2.boundingRect(np.float32([x])) for x in (t1, t2, t))
    if min(r[2], r[3], r1[2], r1[3], r2[2], r2[3]) <= 0:
        return
    t1r = [(t1[i][0] - r1[0], t1[i][1] - r1[1]) for i in range(3)]
    t2r = [(t2[i][0] - r2[0], t2[i][1] - r2[1]) for i in range(3)]
    tr  = [(t[i][0]  - r[0],  t[i][1]  - r[1])  for i in range(3)]
    mask = np.zeros((r[3], r[2], 3), np.float32)
    cv2.fillConvexPoly(mask, np.int32(tr), (1.0, 1.0, 1.0), 16, 0)
    p1 = img1[r1[1]:r1[1] + r1[3], r1[0]:r1[0] + r1[2]]
    p2 = img2[r2[1]:r2[1] + r2[3], r2[0]:r2[0] + r2[2]]
    try:
        M1 = cv2.getAffineTransform(np.float32(t1r), np.float32(tr))
        M2 = cv2.getAffineTransform(np.float32(t2r), np.float32(tr))
        w1 = cv2.warpAffine(p1, M1, (r[2], r[3]), None, cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_REFLECT_101)
        w2 = cv2.warpAffine(p2, M2, (r[2], r[3]), None, cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_REFLECT_101)
    except cv2.error:
        return
    blended = (1.0 - alpha) * w1 + alpha * w2
    roi = out[r[1]:r[1] + r[3], r[0]:r[0] + r[2]]
    out[r[1]:r[1] + r[3], r[0]:r[0] + r[2]] = roi * (1 - mask) + blended * mask


def morph_faces(img1, img2, alpha=MORPH_ALPHA):
    # دمج وجهين في صورة واحدة؛ None عند فشل اكتشاف الوجه
    img1 = cv2.resize(img1, MORPH_SIZE)
    img2 = cv2.resize(img2, MORPH_SIZE)
    p1, p2 = get_landmarks(img1), get_landmarks(img2)
    if p1 is None or p2 is None:
        return None
    avg = (1 - alpha) * p1 + alpha * p2
    try:
        tris = Delaunay(avg).simplices
    except Exception:
        return None
    out = np.zeros(img1.shape, np.float32)
    f1, f2 = img1.astype(np.float32), img2.astype(np.float32)
    for s in tris:
        _morph_triangle(f1, f2, out, p1[s], p2[s], avg[s], alpha)
    return np.clip(out, 0, 255).astype(np.uint8)


def sharpness(img_bgr):
    return cv2.Laplacian(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY),
                         cv2.CV_64F).var()

# اختبار سريع على زوج صور
a, b = selected["train"]["images"][:2]
test_m = morph_faces(cv2.imread(os.path.join(IMG_DIR, a)),
                     cv2.imread(os.path.join(IMG_DIR, b)))
print("✅ دوال المورف تعمل — شكل ناتج تجريبي:",
      None if test_m is None else test_m.shape)


### ٣.٣ توليد صور المورف مع المراجعة التلقائية

اختيار زوجين لشخصين مختلفين من نفس القسم ← دمج ← **مراجعة تلقائية** (وجه واضح + حدة كافية) ← حفظ وتسجيل. مع حد أقصى للمحاولات كي لا تعلق الحلقة.


In [ ]:
LABELS_CSV = os.path.join(RAW_DIR, "labels.csv")

# استئناف تلقائي: إذا كانت المجموعة مولّدة كاملة (قبل إعادة تشغيل الجلسة مثلاً) نتخطى
RESUME = False
if os.path.exists(LABELS_CSV):
    prev = pd.read_csv(LABELS_CSV)
    counts_ok = all(
        (prev[(prev.split == s) & (prev.label == l)].shape[0] >= SPLITS[s][l])
        for s in SPLITS for l in ["real", "morph"])
    files_ok = all(
        len(glob.glob(os.path.join(RAW_DIR, s, l, "*.jpg"))) >= SPLITS[s][l]
        for s in SPLITS for l in ["real", "morph"])
    if counts_ok and files_ok:
        RESUME = True
        print("⏭️ المجموعة مولّدة مسبقاً — تخطي التوليد (استئناف تلقائي)")

records, rejected = [], 0

for split, tgt in ([] if RESUME else list(SPLITS.items())):
    pool = selected[split]["images"]
    out_dir = os.path.join(RAW_DIR, split, "morph")
    os.makedirs(out_dir, exist_ok=True)
    done, attempts = 0, 0
    max_attempts = tgt["morph"] * 30           # حارس ضد الحلقة اللانهائية
    pbar = tqdm(total=tgt["morph"], desc=f"توليد مورف {split}")
    while done < tgt["morph"] and attempts < max_attempts:
        attempts += 1
        a, b = random.sample(pool, 2)
        if img2id[a] == img2id[b]:
            continue
        i1 = cv2.imread(os.path.join(IMG_DIR, a))
        i2 = cv2.imread(os.path.join(IMG_DIR, b))
        if i1 is None or i2 is None:
            continue
        m = morph_faces(i1, i2)
        if m is None or get_landmarks(m) is None or sharpness(m) < MIN_SHARPNESS:
            rejected += 1
            continue
        fn = f"morph_{split}_{done:05d}.jpg"
        cv2.imwrite(os.path.join(out_dir, fn), m,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        records.append({"filename": fn, "split": split, "label": "morph",
                        "src1": a, "src2": b,
                        "id1": img2id[a], "id2": img2id[b]})
        done += 1
        pbar.update(1)
    pbar.close()

if not RESUME:
    n_morph = sum(1 for r in records if r["label"] == "morph")
    print(f"✅ تم توليد {n_morph} صورة مورف — رُفض {rejected} في المراجعة التلقائية")


### ٣.٤ نسخ الصور الحقيقية + الملصقات + المراجعة البصرية والتحقق


In [ ]:
for split, tgt in ([] if RESUME else list(SPLITS.items())):
    out_dir = os.path.join(RAW_DIR, split, "real")
    os.makedirs(out_dir, exist_ok=True)
    for fn in tqdm(selected[split]["images"], desc=f"نسخ حقيقية {split}"):
        img = cv2.imread(os.path.join(IMG_DIR, fn))
        if img is None:
            continue
        out_name = f"real_{fn}"
        cv2.imwrite(os.path.join(out_dir, out_name),
                    cv2.resize(img, MORPH_SIZE),
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        records.append({"filename": out_name, "split": split, "label": "real",
                        "src1": fn, "src2": None,
                        "id1": img2id[fn], "id2": None})

if RESUME:
    labels_df = pd.read_csv(LABELS_CSV)          # استئناف من الملصقات المحفوظة
else:
    labels_df = pd.DataFrame(records)
    labels_df.to_csv(LABELS_CSV, index=False)
print(labels_df.groupby(["split", "label"]).size())

# التحقق: الأعداد + فصل الهويات (بما فيها مصادر المورف)
def ids_of(s):
    sub = labels_df[labels_df.split == s]
    return set(sub.id1.dropna()) | set(sub.id2.dropna())

for i, a in enumerate(SPLIT_NAMES):
    for b in SPLIT_NAMES[i + 1:]:
        assert not (ids_of(a) & ids_of(b)), f"تداخل هويات بين {a} و {b}!"
print("✅ لا توجد أي هوية مشتركة بين الأقسام (شاملاً مصادر المورف)")

# مراجعة بصرية: صف حقيقي وصف مورف
fig, axes = plt.subplots(2, 6, figsize=(16, 5.5))
for row, lab in enumerate(["real", "morph"]):
    sample = labels_df[labels_df.label == lab].sample(6, random_state=SEED)
    for i, (_, r) in enumerate(sample.iterrows()):
        p = os.path.join(RAW_DIR, r.split, r.label, r.filename)
        axes[row, i].imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
        axes[row, i].set_title(f"{'حقيقية' if lab=='real' else 'مدموجة'} ({r.split})",
                               fontsize=9)
        axes[row, i].axis("off")
plt.tight_layout()
plt.show()


## الجزء ٤ — المعالجة المسبقة: 528×528 + CLAHE

- تغيير الحجم إلى **528×528** (متطلب EfficientNet-B6).
- **CLAHE** على قناة الإضاءة L في فضاء LAB — تحسين التباين وإبراز الملامح الدقيقة **دون ضوضاء**.
- حفظ بضغط JPEG موحّد — **نفس الخطوات للحقيقي والمدموج**.


In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))


def standardize(img_bgr):
    img = cv2.resize(img_bgr, (INPUT_SIZE, INPUT_SIZE),
                     interpolation=cv2.INTER_CUBIC)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a2, b2 = cv2.split(lab)
    return cv2.cvtColor(cv2.merge((clahe.apply(l), a2, b2)), cv2.COLOR_LAB2BGR)


# مثال قبل/بعد
sample_p = os.path.join(RAW_DIR, "train", "real",
                        labels_df[(labels_df.split == "train") &
                                  (labels_df.label == "real")].iloc[0].filename)
orig = cv2.imread(sample_p)
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].imshow(cv2.cvtColor(cv2.resize(orig, (INPUT_SIZE, INPUT_SIZE)),
                            cv2.COLOR_BGR2RGB))
axes[0].set_title("قبل (تغيير حجم فقط)")
axes[1].imshow(cv2.cvtColor(standardize(orig), cv2.COLOR_BGR2RGB))
axes[1].set_title("بعد CLAHE — ملامح أوضح")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

# تطبيق على كل الصور
for split in SPLIT_NAMES:
    for cls in CLASSES:
        src = os.path.join(RAW_DIR, split, cls)
        dst = os.path.join(PROC_DIR, split, cls)
        os.makedirs(dst, exist_ok=True)
        for p in tqdm(sorted(glob.glob(os.path.join(src, "*.jpg"))),
                      desc=f"معالجة {split}/{cls}"):
            dstp = os.path.join(dst, os.path.basename(p))
            if os.path.exists(dstp):             # استئناف: تخطي المعالَج مسبقاً
                continue
            img = cv2.imread(p)
            if img is not None:
                cv2.imwrite(dstp, standardize(img),
                            [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])

total = len(glob.glob(os.path.join(PROC_DIR, "*", "*", "*.jpg")))
print(f"✅ تمت معالجة {total} صورة (528×528 + CLAHE + ضغط JPEG)")


### زيادة البيانات (للتدريب فقط)

قلب أفقي، سطوع/تباين، ضبابية، ضوضاء، إعادة ضغط JPEG — تُستخدم في الضبط الدقيق (إن فُعّل) وتحاكي ظروف الصور الواقعية.


In [ ]:
def augment(img):
    if random.random() < 0.5:
        img = cv2.flip(img, 1)
    if random.random() < 0.5:
        img = cv2.convertScaleAbs(img, alpha=random.uniform(0.8, 1.2),
                                  beta=random.uniform(-25, 25))
    if random.random() < 0.3:
        k = random.choice([3, 5, 7])
        img = cv2.GaussianBlur(img, (k, k), 0)
    if random.random() < 0.3:
        noise = np.random.normal(0, random.uniform(5, 15), img.shape)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    if random.random() < 0.4:
        ok, enc = cv2.imencode(".jpg", img,
                               [cv2.IMWRITE_JPEG_QUALITY, random.randint(40, 90)])
        if ok:
            img = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return img


sample = cv2.imread(glob.glob(os.path.join(PROC_DIR, "train", "morph", "*.jpg"))[0])
fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
axes[0].imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
axes[0].set_title("الأصل")
for i in range(1, 6):
    axes[i].imshow(cv2.cvtColor(augment(sample.copy()), cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"معزّزة {i}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## الجزء ٥ — استخراج السمات العميقة بـ EfficientNet-B6

الأساس الرياضي: $F = f_{B6}(I;\theta)$ حيث كل كتلة تنفّذ $X_l=\sigma(W_l * X_{l-1}+b_l)$ بتفعيل Swish، ثم GAP: $F=\frac{1}{N}\sum_i X_L^{(i)}$ فينتج متجه $F\in\mathbb{R}^{2304}$.

- حذف طبقة التصنيف (`include_top=False`) ← الشبكة مستخرج سمات فقط.
- **المرحلة ١**: طبقات مجمّدة بالكامل.
- **المرحلة ٢ (اختيارية — `DO_FINETUNE`)**: فتح block7 والطبقات العلوية فقط بتعلم بطيء.

> 🛠️ **إن ظهر خطأ CUDA هنا (مثل `CUDA_ERROR_INVALID_HANDLE`)**: سياق GPU تعطّل بعد ساعات المعالجة الطويلة — الحل: ‏**Runtime ← Restart session** ثم ‏**Run all**. بفضل **الاستئناف التلقائي** ستتخطى الأجزاء ١–٤ كل ما أُنجز (المورفات المولّدة والصور المعالجة والسمات المحفوظة) خلال ثوانٍ، وتصل إلى هذه الخلية بجلسة GPU نظيفة دون خسارة أي عمل.


In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB6
from tensorflow.keras.applications.efficientnet import preprocess_input

AUTOTUNE = tf.data.AUTOTUNE
EXTRACT_BATCH = 8

base = EfficientNetB6(include_top=False, weights="imagenet",
                      input_shape=(INPUT_SIZE, INPUT_SIZE, 3))
inputs = tf.keras.Input((INPUT_SIZE, INPUT_SIZE, 3))
x = base(inputs, training=False)
gap = layers.GlobalAveragePooling2D(name="gap")(x)          # معادلة GAP
feat_model = Model(inputs, gap, name="dmorphnet_features")
base.trainable = False

print(f"طول متجه السمات F: {feat_model.output_shape[-1]}")


def list_files(split):
    paths, labs = [], []
    for cls, lab in CLASSES.items():
        fs = sorted(glob.glob(os.path.join(PROC_DIR, split, cls, "*.jpg")))
        paths += fs
        labs += [lab] * len(fs)
    return paths, np.array(labs, np.int32)


def extract_ds(paths):
    ds = tf.data.Dataset.from_tensor_slices(list(paths))

    def load(p):
        img = tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
        img = tf.cast(img, tf.float32)
        img = tf.ensure_shape(img, [INPUT_SIZE, INPUT_SIZE, 3])
        return preprocess_input(img)

    return ds.map(load, num_parallel_calls=AUTOTUNE)\
             .batch(EXTRACT_BATCH).prefetch(AUTOTUNE)


F, y01, paths_all = {}, {}, {}
for split in SPLIT_NAMES:
    paths, labs = list_files(split)
    paths_all[split], y01[split] = paths, labs
    npz_path = os.path.join(FEAT_DIR, f"effb6_{split}.npz")
    if os.path.exists(npz_path):
        d = np.load(npz_path)
        if d["X"].shape[0] == len(paths):      # استئناف: سمات محفوظة مسبقاً
            F[split] = d["X"]
            print(f"⏭️ {split}: سمات محفوظة {F[split].shape} — تخطي الاستخراج")
            continue
    print(f"استخراج سمات {split} ({len(paths)} صورة) ...")
    F[split] = feat_model.predict(extract_ds(paths), verbose=1)
    np.savez_compressed(npz_path, X=F[split], y=labs)
    print(f"  {split}: {F[split].shape}")

y = {s: np.where(y01[s] == 0, -1, +1) for s in SPLIT_NAMES}   # -1 حقيقية/+1 مدموجة
print("✅ اكتمل استخراج السمات (طبقات مجمّدة)")


### الضبط الدقيق الاختياري للطبقات العليا (`DO_FINETUNE = True` لتفعيله)

يفتح block7 والطبقات العلوية فقط (وBatchNorm تبقى مجمّدة) بمعدل تعلم 1e-5، ثم يعيد استخراج السمات.


In [ ]:
if DO_FINETUNE:
    drop = layers.Dropout(0.3)(gap)
    out_head = layers.Dense(1, activation="sigmoid", dtype="float32")(drop)
    clf_model = Model(inputs, out_head)

    base.trainable = True
    for layer in base.layers:
        layer.trainable = (layer.name.startswith(("block7", "top"))
                           and not isinstance(layer, layers.BatchNormalization))

    def ft_ds(split, training=False):
        p, labs = paths_all[split], y01[split]
        ds = tf.data.Dataset.from_tensor_slices((list(p), labs))
        if training:
            ds = ds.shuffle(len(p), seed=SEED)

        def load(pp, ll):
            img = tf.io.decode_jpeg(tf.io.read_file(pp), channels=3)
            img = tf.cast(img, tf.float32)
            img = tf.ensure_shape(img, [INPUT_SIZE, INPUT_SIZE, 3])
            if training:
                img = tf.image.random_flip_left_right(img)
                img = tf.image.random_brightness(img, 25.0)
                img = tf.clip_by_value(img, 0.0, 255.0)
            return preprocess_input(img), ll

        return ds.map(load, num_parallel_calls=AUTOTUNE).batch(4).prefetch(AUTOTUNE)

    clf_model.compile(tf.keras.optimizers.Adam(1e-5),
                      "binary_crossentropy", metrics=["accuracy"])
    hist = clf_model.fit(ft_ds("train", True), epochs=2,
                         validation_data=ft_ds("val"))

    for split in SPLIT_NAMES:                      # إعادة الاستخراج بعد الضبط
        F[split] = feat_model.predict(extract_ds(paths_all[split]), verbose=1)
    print("✅ اكتمل الضبط الدقيق وإعادة استخراج السمات")
else:
    print("⏭️ تم تخطي الضبط الدقيق (DO_FINETUNE = False) — نستخدم السمات المجمّدة")


## الجزء ٦ — التصنيف الهجين بـ SVM

معادلات الورقة: الملصقات $y_i\in\{-1,+1\}$، الحد الفاصل $w^TF+b=0$، مسألة التحسين
$\min \frac{1}{2}\lVert w\rVert^2 + C\sum\xi_i$ بقيد $y_i(w^TF_i+b)\ge 1-\xi_i$، والقرار $\hat{y}=\operatorname{sign}(w^TF+b)$.

نوحّد المقاييس، نضبط $C$ على **التحقق**، ندرّب النهائي، ونتحقق من دالة القرار يدوياً.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             roc_curve, auc)

scaler = StandardScaler().fit(F["train"])
Fs = {s: scaler.transform(F[s]) for s in SPLIT_NAMES}

# ضبط C على مجموعة التحقق
results_C = {}
for C in [0.01, 0.1, 1.0, 10.0]:
    clf = LinearSVC(C=C, random_state=SEED).fit(Fs["train"], y["train"])
    results_C[C] = accuracy_score(y["val"], clf.predict(Fs["val"]))
    print(f"C = {C:<6} → دقة التحقق = {results_C[C]:.4f}")
BEST_C = max(results_C, key=results_C.get)

svm_final = LinearSVC(C=BEST_C, random_state=SEED).fit(Fs["train"], y["train"])
w, b = svm_final.coef_[0], float(svm_final.intercept_[0])
print(f"\n🏆 أفضل C = {BEST_C} — أبعاد w: {w.shape} ، b = {b:.4f}")

# التحقق اليدوي: sign(wF+b) يطابق predict تماماً
scores_test = Fs["test"] @ w + b
assert (np.sign(scores_test) == svm_final.predict(Fs["test"])).all()
print("✅ تطابق sign(wᵀF+b) مع predict بنسبة 100%")

plt.figure(figsize=(9, 4))
plt.hist(scores_test[y["test"] == -1], bins=40, alpha=0.6, label="حقيقية (-1)")
plt.hist(scores_test[y["test"] == +1], bins=40, alpha=0.6, label="مدموجة (+1)")
plt.axvline(0, color="black", linestyle="--", label="الحد الفاصل wᵀF+b=0")
plt.xlabel("wᵀF + b")
plt.ylabel("عدد الصور")
plt.title("فصل الفئتين في فضاء السمات (اختبار)")
plt.legend()
plt.tight_layout()
plt.show()


## الجزء ٧ — النتائج: مصفوفة الالتباس + المقاييس + ROC/AUC

مصفوفة الالتباس بأسلوب الشكل ٢، وقيم TP/TN/FP/FN، والمقاييس الأربعة، ومنحنى ROC بأسلوب الشكل ٣ (في الورقة: دقة 89.9% وAUC = 0.965 على المجموعة الكاملة).


In [ ]:
pred_test = svm_final.predict(Fs["test"])
cm = confusion_matrix(y["test"], pred_test, labels=[-1, +1])
TN, FP, FN, TP = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5))

# مصفوفة الالتباس (أسلوب الشكل 2)
im = axes[0].imshow(cm, cmap="Blues")
plt.colorbar(im, ax=axes[0])
axes[0].set_title("Confusion Matrix")
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Real", "Morph"])
axes[0].set_yticklabels(["Real", "Morph"])
axes[0].set_xlabel("Predicted label"); axes[0].set_ylabel("True label")
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha="center", va="center", fontsize=14,
                     color="white" if cm[i, j] > cm.max() / 2 else "black")

# منحنى ROC (أسلوب الشكل 3)
fpr, tpr, thresholds = roc_curve(y["test"], scores_test, pos_label=+1)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color="tab:blue", linewidth=2.2,
             label=f"AUC={roc_auc:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="tab:orange", linewidth=1.8)
axes[1].set_title("ROC Curve")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].grid(alpha=0.35)
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

metrics = {
    "Accuracy":  accuracy_score(y["test"], pred_test),
    "Precision": precision_score(y["test"], pred_test, pos_label=+1),
    "Recall":    recall_score(y["test"], pred_test, pos_label=+1),
    "F1-Score":  f1_score(y["test"], pred_test, pos_label=+1),
    "AUC":       roc_auc,
}
print(f"TP={TP}  TN={TN}  FP={FP}  FN={FN}")
print(f"FN (مورف فائت) هو الأخطر بيومترياً — FP يُحل بالمراجعة اليدوية\n")
for k, v in metrics.items():
    print(f"  {k:10s} = {v:.4f}")
print(f"\n(المرجع في الورقة: الدقة 89.9% ، AUC = 0.965 على 45,000 صورة)")
print(classification_report(y["test"], pred_test,
                            target_names=["حقيقية (Real)", "مدموجة (Morph)"]))


## الجزء ٨ — تحسين العتبة

القرار: مدموجة إذا $s \ge \tau$. نمسح قيم $\tau$ على **التحقق**، نختار ما يعظّم F1 (التوازن بين اكتشاف المورف وتجنّب اتهام الحقيقي)، ثم نقيّم على الاختبار.


In [ ]:
s_val = Fs["val"] @ w + b
taus = np.linspace(s_val.min(), s_val.max(), 300)
f1s = [f1_score(y["val"], np.where(s_val >= t, +1, -1),
                pos_label=+1, zero_division=0) for t in taus]
TAU = float(taus[int(np.argmax(f1s))])

pred_opt = np.where(scores_test >= TAU, +1, -1)
acc_def = accuracy_score(y["test"], pred_test)
acc_opt = accuracy_score(y["test"], pred_opt)
cm_opt = confusion_matrix(y["test"], pred_opt, labels=[-1, +1])

plt.figure(figsize=(9, 4))
plt.plot(taus, f1s, label="F1 على التحقق")
plt.axvline(0, color="gray", linestyle=":", label="الافتراضية τ=0")
plt.axvline(TAU, color="green", linestyle="--", label=f"المختارة τ*={TAU:.3f}")
plt.xlabel("العتبة τ"); plt.ylabel("F1")
plt.title("اختيار العتبة على مجموعة التحقق")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"العتبة المختارة τ* = {TAU:.4f}")
print(f"دقة الاختبار: افتراضية τ=0 → {acc_def:.4f} | "
      f"مختارة τ* → {acc_opt:.4f} ({(acc_opt-acc_def)*100:+.2f} نقطة)")
print(f"FN: {FN} ← {int(cm_opt[1,0])}   |   FP: {FP} ← {int(cm_opt[0,1])}")


## الجزء ٩ — (اختياري) مقارنة المعماريات: الجدول ٢

فعّل `RUN_COMPARISONS = True` في الجزء ١ لتشغيلها: B0 + Softmax (سمات 1280)، B5 + SVM خطي (2048)، مقابل B6 المقترح (2304). المرجع المنشور: 62.4% / 66.13% / **89.9%**.


In [ ]:
if RUN_COMPARISONS:
    from tensorflow.keras.applications import EfficientNetB0, EfficientNetB5

    def extract_with(model_cls, size, paths):
        m = model_cls(include_top=False, weights="imagenet",
                      pooling="avg", input_shape=(size, size, 3))

        def load(p):
            img = tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
            img = tf.image.resize(tf.cast(img, tf.float32), [size, size])
            return preprocess_input(img)

        ds = (tf.data.Dataset.from_tensor_slices(list(paths))
              .map(load, num_parallel_calls=AUTOTUNE).batch(16).prefetch(AUTOTUNE))
        return m.predict(ds, verbose=1)

    comp = {}
    # B0 + Softmax
    b0tr = extract_with(EfficientNetB0, 224, paths_all["train"])
    b0te = extract_with(EfficientNetB0, 224, paths_all["test"])
    sc0 = StandardScaler().fit(b0tr)
    h0 = tf.keras.Sequential([tf.keras.layers.Input((b0tr.shape[1],)),
                              tf.keras.layers.Dense(2, activation="softmax")])
    h0.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
    h0.fit(sc0.transform(b0tr), y01["train"], epochs=10, batch_size=128, verbose=0)
    p0 = np.where(h0.predict(sc0.transform(b0te), verbose=0)[:, 1] >= 0.5, +1, -1)
    comp["B0 + Softmax (1280)"] = accuracy_score(y["test"], p0)

    # B5 + SVM خطي
    b5tr = extract_with(EfficientNetB5, 456, paths_all["train"])
    b5te = extract_with(EfficientNetB5, 456, paths_all["test"])
    sc5 = StandardScaler().fit(b5tr)
    c5 = LinearSVC(C=1.0, random_state=SEED).fit(sc5.transform(b5tr), y["train"])
    comp["B5 + SVM خطي (2048)"] = accuracy_score(y["test"],
                                                 c5.predict(sc5.transform(b5te)))

    comp["B6 المقترح (2304)"] = accuracy_score(y["test"], pred_test)

    print("الجدول 2 — دقة الاختبار (المرجع: 62.4 / 66.13 / 89.9):")
    for k, v in comp.items():
        print(f"  {k:24s}: {v*100:.2f}%")
else:
    print("⏭️ تم تخطي مقارنة المعماريات (RUN_COMPARISONS = False)")
    print("   المرجع المنشور — B0: 62.4% | B5: 66.13% | B6 المقترح: 89.9%")


## الجزء ١٠ — التنبؤ الفوري (الشكلان ٤ و٥)

- ارفع أي صورة جديدة ← معالجة ← سمات B6 ← SVM بالعتبة المحسّنة ← **MORPH IMAGE / REAL IMAGE** بعنوان أحمر مع الثقة.
- يدعم **تعدد الوجوه**: يكتشفها MediaPipe ويصنّف كل وجه (أحمر = مدموج، أخضر = حقيقي).


In [ ]:
# كاشف وجوه موحد يعمل مع نسختي MediaPipe (القديمة solutions والجديدة Tasks)
if hasattr(mp, "solutions"):
    _detector = mp.solutions.face_detection.FaceDetection(
        model_selection=1, min_detection_confidence=0.5)

    def detect_face_boxes(img_bgr):
        h, wd = img_bgr.shape[:2]
        res = _detector.process(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        if not res.detections:
            return []
        out = []
        for det in res.detections:
            bb = det.location_data.relative_bounding_box
            out.append((int(bb.xmin * wd), int(bb.ymin * h),
                        int(bb.width * wd), int(bb.height * h)))
        return out
else:
    import urllib.request
    from mediapipe.tasks.python import BaseOptions
    from mediapipe.tasks.python import vision as mp_vision
    _DET_MODEL = "/content/blaze_face_short_range.tflite"
    if not os.path.exists(_DET_MODEL):
        urllib.request.urlretrieve(
            "https://storage.googleapis.com/mediapipe-models/face_detector/"
            "blaze_face_short_range/float16/1/blaze_face_short_range.tflite",
            _DET_MODEL)
    _detector = mp_vision.FaceDetector.create_from_options(
        mp_vision.FaceDetectorOptions(
            base_options=BaseOptions(model_asset_path=_DET_MODEL),
            min_detection_confidence=0.5))

    def detect_face_boxes(img_bgr):
        rgb = np.ascontiguousarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        res = _detector.detect(mp_img)
        return [(int(d.bounding_box.origin_x), int(d.bounding_box.origin_y),
                 int(d.bounding_box.width), int(d.bounding_box.height))
                for d in res.detections]


def predict_face(img_bgr):
    # المسار الكامل لوجه واحد: معالجة ← سمات ← قرار + ثقة + أزمنة
    t0 = time.perf_counter()
    proc = standardize(img_bgr)
    rgb = cv2.cvtColor(proc, cv2.COLOR_BGR2RGB).astype(np.float32)
    t1 = time.perf_counter()
    feat = feat_model.predict(preprocess_input(rgb)[None, ...], verbose=0)
    t2 = time.perf_counter()
    s = float(scaler.transform(feat) @ w + b)
    conf = 1.0 / (1.0 + np.exp(-(s - TAU)))          # سيغمويد حول العتبة
    t3 = time.perf_counter()
    label = "MORPH IMAGE" if s >= TAU else "REAL IMAGE"
    return label, conf, s, {"pre": t1 - t0, "feat": t2 - t1,
                            "svm": t3 - t2, "total": t3 - t0}


def show_result(img_bgr, label, conf):
    # عرض بأسلوب الشكلين 4 و5: عنوان أحمر فوق الصورة
    plt.figure(figsize=(4.5, 5.5))
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(f"{label}\nConfidence:{conf:.4f}",
              color="red", fontsize=14, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def predict_image(img_bgr):
    # يدعم تعدد الوجوه؛ إن لم يُكتشف وجه يصنّف الصورة كاملة
    h, wd = img_bgr.shape[:2]
    boxes = []
    for (x, yy, bw, bh) in detect_face_boxes(img_bgr):
        m = int(0.25 * max(bw, bh))          # هامش 25% حول الوجه
        boxes.append((max(0, x - m), max(0, yy - m),
                      min(wd, x + bw + m) - max(0, x - m),
                      min(h, yy + bh + m) - max(0, yy - m)))
    if not boxes:
        lab, cf, s, tms = predict_face(img_bgr)
        return [(None, lab, cf, tms)], img_bgr.copy()
    annotated, out = img_bgr.copy(), []
    for (x, yy, bw, bh) in boxes:
        lab, cf, s, tms = predict_face(img_bgr[yy:yy + bh, x:x + bw])
        out.append(((x, yy, bw, bh), lab, cf, tms))
        color = (0, 0, 255) if lab == "MORPH IMAGE" else (0, 200, 0)
        cv2.rectangle(annotated, (x, yy), (x + bw, yy + bh), color, 3)
        cv2.putText(annotated, f"{lab.split()[0]} {cf:.2f}",
                    (x, max(25, yy - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
    return out, annotated


# عرض تجريبي فوري على عينة حقيقية وعينة مدموجة من الاختبار
for cls in ["real", "morph"]:
    p = random.choice(glob.glob(os.path.join(PROC_DIR, "test", cls, "*.jpg")))
    img = cv2.imread(p)
    lab, cf, s, tms = predict_face(img)
    print(f"الحقيقة: {'حقيقية' if cls=='real' else 'مدموجة'} — التنبؤ: {lab} — "
          f"Confidence:{cf:.4f} — {tms['total']*1000:.0f} ms")
    show_result(img, lab, cf)


### 📤 ارفع صورك الخاصة الآن (كما في الشكلين ٤ و٥)


In [ ]:
from google.colab import files

uploaded = files.upload()
for fname in uploaded:
    img = cv2.imdecode(np.frombuffer(uploaded[fname], np.uint8),
                       cv2.IMREAD_COLOR)
    if img is None:
        print(f"⚠️ تعذّرت قراءة {fname}")
        continue
    results, annotated = predict_image(img)
    if len(results) == 1 and results[0][0] is None:
        _, lab, cf, tms = results[0]
        show_result(img, lab, cf)
    else:
        plt.figure(figsize=(8, 8))
        plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        plt.title(f"عدد الوجوه: {len(results)}")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
    for i, (box, lab, cf, tms) in enumerate(results, 1):
        print(f"  وجه {i}: {lab}  Confidence:{cf:.4f}  "
              f"({tms['total']*1000:.0f} ms)")


### قياس سرعة التنبؤ


In [ ]:
sample_paths = glob.glob(os.path.join(PROC_DIR, "test", "*", "*.jpg"))
predict_face(cv2.imread(sample_paths[0]))          # إحماء

agg = {"pre": [], "feat": [], "svm": [], "total": []}
for p in random.sample(sample_paths, min(20, len(sample_paths))):
    _, _, _, tms = predict_face(cv2.imread(p))
    for k in agg:
        agg[k].append(tms[k])

for k, name in [("pre", "المعالجة المسبقة"), ("feat", "سمات B6"),
                ("svm", "قرار SVM"), ("total", "الإجمالي")]:
    print(f"  {name:18s}: {np.mean(agg[k])*1000:7.1f} ms")
print(f"\nمعدل المعالجة: ~{1.0/np.mean(agg['total']):.1f} صورة/ثانية — "
      "القرار بعد السمات شبه لحظي")


## الجزء ١١ — الحفظ في Google Drive (اختياري) والخلاصة

فعّل `SAVE_TO_DRIVE = True` في الجزء ١ لحفظ: النموذج، المصنّف، الـ Scaler، العتبة، السمات، والنتائج.


In [ ]:
import joblib

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/DMorphNet_models"
    os.makedirs(SAVE_DIR, exist_ok=True)
    feat_model.save(os.path.join(SAVE_DIR, "effb6_features.keras"))
    joblib.dump(svm_final, os.path.join(SAVE_DIR, "svm_final.joblib"))
    joblib.dump(scaler, os.path.join(SAVE_DIR, "scaler_final.joblib"))
    np.savez(os.path.join(SAVE_DIR, "optimal_threshold.npz"), tau=TAU)
    for split in SPLIT_NAMES:
        shutil.copy(os.path.join(FEAT_DIR, f"effb6_{split}.npz"), SAVE_DIR)
    print("✅ تم الحفظ في:", SAVE_DIR)
    print(os.listdir(SAVE_DIR))
else:
    print("⏭️ الحفظ في Drive معطّل (SAVE_TO_DRIVE = False)")

print("\n" + "=" * 55)
print("الخلاصة النهائية — D-MorphNet")
print("=" * 55)
print(f"الوضع            : {'تجريبي' if DEMO else 'كامل'}")
print(f"إجمالي الصور     : {len(labels_df)} "
      f"({t_real} حقيقية + {t_morph} مدموجة)")
print(f"المستخرج         : EfficientNet-B6 "
      f"({'مضبوط دقيقاً' if DO_FINETUNE else 'مجمّد'}) — سمات 2304")
print(f"المصنّف          : SVM (C = {BEST_C} ، τ* = {TAU:.4f})")
print(f"دقة الاختبار     : {acc_opt:.4f}")
print(f"AUC              : {roc_auc:.4f}")
print(f"TP/TN/FP/FN      : {TP}/{TN}/{FP}/{FN}")
print("\n🎉 اكتمل خط أنابيب D-MorphNet كاملاً في ملف واحد")
